<a id='toc'></a>
# Содержание
* [Описание системы](#system_description)
* [Генерирование данных](#generating_data)
* [EKF-фильтр 1](#ekf_filter_1)
* [EKF-фильтр 2](#ekf_filter_2)

In [ ]:
import copy
import typing as T
import numbers
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

<a id='system_description'></a>
## Описание системы<sup>[toc](#toc)</sup>

В данной домашней работе рассмотрим процесс восстановления скрытого состояния колеблющегося маятника. Истинная (но условно неизвестная нам) траектория процесса описывается законом:
$$
x(t) = A \cos(2 \pi f t + \phi_0),
$$
где:
* $A$ &mdash; амплитуда колебаний
* $f$ &mdash; частота колебаний
* $\varphi_0$ &mdash; начальая фаза колебаний, $\varphi_0 \in [0, 2\pi)$

Для удобства вместо частоты $f$ рассматривают угловую частоту (или скорость) $\omega = 2\pi f$:
$$
x(t) = A \cos(\omega t + \varphi_0)
$$

В некоторые моменты времени $t_1, \dots, t_N, \dots$ мы наблюдаем положения маятника $z_1, \dots, z_N, \dots$, которые, однако, получены с некоторой погрешностью, т.е. с шумом. Иными словами 
$$
z_i = x(t_i) + \varepsilon_i,
$$
где $\varepsilon_i$ &mdash; шум наблюдения. В рассматриваемой далее модели будем считать шум нормальным, т.е. $\varepsilon_i \sim \mathcal{N}(0, \sigma_\varepsilon^2)$.

Нашей задачей является получение оценки $\hat{x}(t)$ траектории маятника на основании наблюдений $z_1, \dots, z_N, \dots$ с помощью фильтра Калмана (обычного или расширенного в зависимости от конкретной модели системы).

<a id='generating_data'></a>
## Генерирование данных<sup>[toc](#toc)</sup>

Мы будем иметь дело с тремя типами данных:
* **gt data**
* **noisy observed data**
* **ekf data**

> gt &mdash; сокращение от ground truth, т.е. истинные значения

#### Gt-параметры маятника<sup>[toc](#toc)</sup>

In [ ]:
gt_amplitude = 2    # [m]eters
gt_frequency = 0.5  # [Hz]
gt_angular_frequency = 2 * np.pi * gt_frequency
gt_initial_phase = np.pi / 4

print(f'GT amplitude: {gt_amplitude} [m]')
print(f'GT frequency: {gt_frequency} [Hz]')
print(f'GT angular frequency: {gt_angular_frequency} [rad/s] ({np.rad2deg(gt_angular_frequency)} [deg/s])')
print(f'GT initial phase: {gt_initial_phase} [rad] ({np.rad2deg(gt_initial_phase)} [deg])')

# Задаем границы времени моделирования системы
start_time = 0.
finish_time = 100.

# Задаем частоту наблюдений. Тут специально выбрано значение не кратное периоду колебаний
# Дополнительно задаем равномерный случайный шум моментов наблюдений
observation_frequency = 9  # [Hz]
observation_period_mean = 1 / observation_frequency  # [s]seconds
observation_period_var = (0.3 * observation_period_mean) ** 2
observation_period_distr_type = "uniform"
print(f'GT observation period: {observation_period_mean} [s]')
print(f'GT observation period var: {observation_period_var} [s^2]')

# Шум наблюдений (стандартное отклонение)
gt_observation_noise_std = 0.5

# Моменты наблюдений
random_seed = 346276
random_state = np.random.RandomState(random_seed)
observations_times = np.arange(start_time, finish_time, observation_period_mean)

if observation_period_var > 0:
    if observation_period_distr_type == "uniform":
        span = np.sqrt(observation_period_var * 12) / 2
        assert span < observation_period_mean
        observations_times += random_state.uniform(low=-span, high=span, size=len(observations_times))
        del span
    elif observation_period_distr_type == "normal":
        observations_times += random_state.normal(scale=np.sqrt(observation_period_var), size=len(observations_times))
    else:
        # Нет шума моментов наблюдений
        assert (observation_period_distr_type is None) or (observation_period_distr_type == "none")
print(f'Number of observations: {len(observations_times)}')
print(f'First observation time: {observations_times[0]} [s]')
print(f'Last observation time:  {observations_times[-1]} [s]')
print(f'Estimated observation period mean: {np.mean(np.diff(observations_times))} [s]')

#### Генерирование GT-наблюдений<sup>[toc](#toc)</sup>

In [ ]:
gt_observations = gt_amplitude * np.cos(gt_angular_frequency * observations_times + gt_initial_phase)

# Отрисуем N первых наблюдений (если отрисовать все, то будет "каша")
num_observations_to_draw = 100
plt.plot(
    observations_times[:num_observations_to_draw],
    gt_observations[:num_observations_to_draw],
    linestyle=None, marker='o', color='b')
plt.grid(which='both', linestyle='--', alpha=0.5)

#### Генерирование зашумленных наблюдений<sup>[toc](#toc)</sup>

In [ ]:
random_state = np.random.RandomState(124235)

observations = gt_observations + random_state.normal(
    loc=0,
    scale=gt_observation_noise_std,
    size=gt_observations.shape)

plt.plot(
    observations_times[:num_observations_to_draw],
    gt_observations[:num_observations_to_draw], linestyle=None, marker='o', color='b', label='gt')
plt.plot(
    observations_times[:num_observations_to_draw],
    observations[:num_observations_to_draw], linestyle=None, marker='x', color='r', label='noisy')
plt.grid(which='both', linestyle='--', alpha=0.5)
plt.legend();

<a id='ekf_filter_1'></a>
## 1. EKF-фильтрация для состояния $(x, \upsilon, a)^T$<sup>[toc](#toc)</sup>

В качестве вектора состояния фильтра Калмана рассматриваем следующий вектор:
$$
\boldsymbol{x}(t) =
\begin{pmatrix}
x(t) \\
\upsilon(t) \\
a(t)
\end{pmatrix},
$$
где
* $x(t)$ &mdash; координата X маятника
* $\upsilon(t)$ &mdash; проекция скорости на ось X
* $a(t)$ &mdash; проекция ускорения на ось X

Далее нужно задать модель эволюции/предсказания системы и модель наблюдений.

### 1.1 Модель эволюции<sup>[toc](#toc)</sup>

**TODO. Нужно найти матрицу перехода $A(t, \Delta t)$ в выражения ниже:**

$$
\boldsymbol{x}(t + \Delta t) = f(\boldsymbol{x}(t)) \approx A(t, \Delta t) \boldsymbol{x}(t)
$$

$$
A(t, \Delta t) = ?
$$

### 1.2 Модель наблюдений<sup>[toc](#toc)</sup>
**TODO. Нужно найти матрицу наблюдений $C(t)$ в выражении ниже**:
$$
z(t) = C(t) \boldsymbol{x}(t) + \varepsilon(t)
$$

$$
C(t) = ?
$$

### 1.3 Моделирование<sup>[toc](#toc)</sup>
Теперь когда задали обе основные модели необходимые для запуска фильтра Калмана, перейдем непосредственно к процессу фильтрации..

In [ ]:
# Для удобства создадим отдельный класс для хранения EKF-состояния
from collections import namedtuple

EkfState = namedtuple('EkfState', ['time', 'size', 'mean', 'cov'])

#### Инициализируем начальное состояние фильтра Калмана<sup>[toc](#toc)<sup>

In [ ]:
def get_initial_ekf_state(initial_time: T.Optional[float] = None) -> EkfState:
    # Функциональный генератор начального состояния EKF-фильтра
    # Если нужно, поменайте значения начального среднего и ковариации на свое усмотрение
    if initial_time is None:
        initial_time = 0.0
    assert isinstance(initial_time, numbers.Number)
    return EkfState(
        time=float(initial_time),
        size=3,
        mean=np.zeros(3),
        cov=np.diag([10., 10., 10.]))


initial_ekf_state = get_initial_ekf_state()
print('Initial EKF state:')
print(f'\ttime: {initial_ekf_state.time}')
print(f'\tsize: {initial_ekf_state.size}')
print(f'\tmean: {initial_ekf_state.mean}')
print(f'\tcov: {initial_ekf_state.cov[np.arange(3), np.arange(3)]}')

#### Матрицы перехода и наблюдений<sup>[toc](#toc)<sup>

In [ ]:
def get_ekf_transition_matrix(
        ekf_state: EkfState,
        time_step: float) -> np.ndarray:
    """
    В наиболее общем случае матрица перехода из момента времени t в момент времени t + dt
    может зависить от текущего времени t, от размера шага перехода dt, а также от среднего mu.

    В простейших случаях возвращаемая матрица перехода может вообще быть константной или
    зависеть только от dt. Данная функция представляет наиболее общий интерфейс
    
    Должна вернуть матрицу размера (ekf_state.size, ekf_state.size)
    """
    # TODO(Implement)
    raise NotImplementedError("Please implement this function")


def get_ekf_transition_noise_cov_matrix(ekf_state: EkfState, time_step: float):
    # TODO(Implement)
    # Так как в задаче переменный шаг, то для получения хорошего результата матрицу шума скорее всего
    # потребуется сделать линейно зависимой от шага. Т.е. получается так, что дисперсия перехода
    # в новое состояние как бы нарастает с некоторой скоростью: чем больше не видим систему,
    # тем больше становится дисперсия её состояния
    raise NotImplementedError("Please implement this function")


def get_ekf_expected_observation(ekf_state: EkfState) -> np.ndarray:
    """
    В случае, когда наблюдение является линейной функцией состояния, данная функция по сути
    возвращает просто C * x. Но в случае нелинейных налюдений ожидаемое значение наблюдения в некотором
    EKF-состоянии x придется считать отдельно
    """
    # TODO(Implement)
    raise NotImplementedError("Please implement this function")


def get_ekf_observation_matrix(ekf_state: EkfState) -> np.ndarray:
    """
    То же самое, что и предыдущая функция, но теперь для матрицы наблюдений.
    В нашем случае у нас только один тип наблюдений - это координата x, а потому и генератор только один.
    Если бы было N типов наблюдений, то и функций-генераторов было бы тоже N
    
    Должна вернуть матрицу размера (observatoin_size, ekf_state_size)
    """
    # TODO(Implement)
    raise NotImplementedError("Please implement this function")


def get_ekf_observation_noise_cov_matrix(ekf_state: EkfState) -> np.ndarray:
    """
    Вернуть матрицу шума наблюдений
    """
    # TODO(Implement)
    raise NotImplementedError("Please implement this function")

In [ ]:
def kalman_transit_state(ekf_state: EkfState, target_time: float) -> EkfState:
    """
    Внутри этой функции вам потребуется вызвать
    * get_ekf_transition_matrix
    * get_ekf_transition_noise_cov_matrix
    """
    assert ekf_state.time <= target_time
    if ekf_state.time == target_time:
        # Nothing to do
        return ekf_state
    time_step = target_time - ekf_state.time

    # TODO(Implement)
    raise NotImplementedError("Please implement this function")


def kalman_process_observation(ekf_state: EkfState, observation: np.ndarray) -> EkfState:
    assert isinstance(ekf_state, EkfState)
    assert isinstance(observation, np.ndarray)

    # TODO(Implement)
    raise NotImplementedError("Please implement this function")

#### Запускаем моделирование<sup>[toc](#toc)<sup>
    
Шагаем последовательно по моментам наблюдения координаты $x$:
1. При обработке очередного наблюдения сначала делаем предсказание состояния на момент наблюдения $t$
2. Затем проводим обработку наблюдения в момент $t$

In [ ]:
def get_initial_ekf_state(initial_time: T.Optional[float] = None) -> EkfState:
    # Функциональный генератор начального состояния EKF-фильтра
    # Если нужно, поменайте значения начального среднего и ковариации на свое усмотрение
    if initial_time is None:
        initial_time = 0.0
    assert isinstance(initial_time, numbers.Number)
    return EkfState(
        time=float(initial_time),
        size=3,
        mean=np.zeros(3),
        cov=np.diag([10., 10., 10.]))


initial_ekf_state = get_initial_ekf_state(initial_time=observations_times[0])
print('Initial EKF state:')
print(f'\ttime: {initial_ekf_state.time}')
print(f'\tsize: {initial_ekf_state.size}')
print(f'\tmean: {initial_ekf_state.mean}')
print(f'\tcov: {initial_ekf_state.cov[np.arange(3), np.arange(3)]}')

In [ ]:
ekf_state = copy.deepcopy(initial_ekf_state)
ekf_states = []

assert len(observations_times) == len(observations)
for observation_time, observation in zip(observations_times, observations):
    assert ekf_state.time <= observation_time, "EKF state time {} must not exceed next observation time {}".format(
        ekf_state.time, observation_time
    )
    # Step 1: Transition/prediction
    # moving state to current observation time that is finding
    # prior state distribution for time "observation_time"
    ekf_state = kalman_transit_state(ekf_state, observation_time)
    
    # Step 2. State correction/observation processing
    # find posterior state distribution for time "observation time"
    observation = np.array([observation], dtype=np.float64)
    ekf_state = kalman_process_observation(ekf_state, observation)

    ekf_states.append(ekf_state)
    
xva_ekf_states = copy.deepcopy(ekf_states)

### Визуализация результатов оценки<sup>[toc](#toc)</sup>

Заранее отметим, что скорее всего у вас не выйдет получить хорошие оценки на скорость и ускорение. Основная причина этому в том, что нет источников информации, которые бы непосредственно наблюдали за этими компонентами вектора состояния. Например скорость $\upsilon$ фактически наблюдает колёсная одометрия, а ускорение $a$ &mdash; аккселерометр. Но в рассмотренной выше модели системы в нашем распоряжении есть только виртуальный GNSS-сенсор, который позволяет наблюдать за координатой $x$.

In [ ]:
# Отрисуем последние 100 значений
num_observations_to_draw = 100

fig, axarr = plt.subplots(nrows=3, ncols=1, sharex=True, figsize=(8, 15))

# Drawing x(t)
gt_xs = gt_observations
obs_xs = observations
ekf_xs = [ekf_state.mean[0] for ekf_state in xva_ekf_states]

ax = axarr[0]
ax.plot(
    observations_times[-num_observations_to_draw:],
    gt_xs[-num_observations_to_draw:], color='g', label='gt')
ax.plot(
    observations_times[-num_observations_to_draw:],
    obs_xs[-num_observations_to_draw:], color='r', label='obs')
ax.plot(
    observations_times[-num_observations_to_draw:],
    ekf_xs[-num_observations_to_draw:], color='b', label='ekf')
ax.set_title('x(t)')
ax.set_ylabel('m')
ax.legend()

# Drawing v(t)
ax = axarr[1]
gt_velocities = -gt_angular_frequency * gt_amplitude * np.sin(
    gt_angular_frequency * observations_times + gt_initial_phase)
ekf_velocities = [ekf_state.mean[1] for ekf_state in xva_ekf_states]

ax.plot(
    observations_times[-num_observations_to_draw:],
    gt_velocities[-num_observations_to_draw:], color='g', label='gt')
ax.plot(
    observations_times[-num_observations_to_draw:],
    ekf_velocities[-num_observations_to_draw:], color='b', label='ekf')

ax.set_title('v(t)')
ax.set_ylabel('m/s')
ax.legend()

# Drawing a(t)
ax = axarr[2]
gt_accelerations = -gt_angular_frequency**2 * gt_amplitude * np.cos(
    gt_angular_frequency * observations_times + gt_initial_phase)
ekf_accelerations = [ekf_state.mean[2] for ekf_state in xva_ekf_states]

ax.plot(
    observations_times[-num_observations_to_draw:],
    gt_accelerations[-num_observations_to_draw:], color='g', label='gt')
ax.plot(
    observations_times[-num_observations_to_draw:],
    ekf_accelerations[-num_observations_to_draw:], color='b', label='ekf')

ax.set_title('a(t)')
ax.set_ylabel('m/s^2')
ax.legend();

### Подсчет MSE<sup>[toc](#toc)</sup>

In [ ]:
np.mean((gt_xs - ekf_xs)**2)

<a id='ekf_filter_2'></a>
## 2. EKF-фильтрация для состояние $(\varphi, A, \omega)^T$<sup>[toc](#toc)</sup>

В данном пункте требуется реализовать EKF-фильтрацию для состояния следующего вида:
$$
\boldsymbol{x}(t) =
\begin{pmatrix}
\varphi(t) \\
A(t) \\
\omega(t)
\end{pmatrix},
$$
где
* $\varphi(t)$ &mdash; фаза маятника
* $A(t)$ &mdash; амплитуда колебаний маятника
* $\omega(t)$ &mdash; угловая частота маятника

Тут может возникнуть вопрос, а зачем в состояние помещены величины, которые являются константными? Ну тут дело в том, что это мы знаем, что они константны, а если бы мы просто наюблюдали лишь $z_1$, $z_2$, $\dots$, то по зашумленным данным такого вывода сделать не выйдет. Более того, вполне в реальной системе есть трение, которое будет влиять на амплидуту и частоту колебаний. Плюс возможны внешние воздействия для раскачивания маятника.

Далее нужно задать модель эволюции/предсказания системы и модель наблюдений.

### 2.1 Модель эволюции<sup>[toc](#toc)</sup>

**TODO. Нужно найти матрицу перехода $A(t, \Delta t)$ в выражения ниже:**

$$
\boldsymbol{x}(t + \Delta t) = f(\boldsymbol{x}(t)) \approx A(t, \Delta t) \boldsymbol{x}(t)
$$

$$
A(t, \Delta t) = ?
$$

### 2.2 Модель наблюдений<sup>[toc](#toc)</sup>
В данном случае функция наблюдения $x(t) = A(t) \cos \phi(t)$ нелинейна, и здесь при коррекции состояния на основании наблюдения нужно будет воспользоваться немного другими формулами, чем раньше.

**TODO. Нужно найти якобиан функции наблюдения $g(\cdot)$ в выражении ниже**:
$$
z(t) = g(\boldsymbol{x}(t)) + \varepsilon(t) \approx g(\boldsymbol{\mu}(t)) + J_g(\boldsymbol{\mu}(t)) (\boldsymbol{x}(t) - \boldsymbol{\mu}(t)) + \varepsilon(t)
$$

$$
J_g(\boldsymbol{x}(t)) = ?
$$

В случае нелинейного наблюдения $\boldsymbol{z}(t) = \boldsymbol{g}(\boldsymbol{x}(t))$ формулы коррекции прогноза выглядят следующим образом:
\begin{align*}
&K(t) = \hat{\Sigma}(t) \cdot J_{g}(\hat{\boldsymbol{\mu}}(t))^T \cdot (J_{g}(\hat{\boldsymbol{\mu}}(t)) \cdot \hat{\Sigma}(t) \cdot  J_{g}(\hat{\boldsymbol{\mu}}(t))^T + Q(t))^{-1},\\
&\boldsymbol{\mu}(t) = \hat{\boldsymbol{\mu}}(t) + K(t) (\boldsymbol{z}(t) - \boldsymbol{g}(\hat{\boldsymbol{\mu}}(t)),\\
&\Sigma(t) = (I - K(t) J_{g}(\hat{\boldsymbol{\mu}}(t))) \cdot \hat{\Sigma}(t).
\end{align*}
Здесь два основных изменения:
* Вместо матрицы наблюдения $C(t)$ теперь стоит якобиан $J_{g}(\hat{\boldsymbol{\mu}}(t))$
* Вместо произведения $C(t) \boldsymbol{x}(t)$ теперь стоит $\boldsymbol{g}(\hat{\boldsymbol{\mu}}(t))$. Заметим, что в обоих случаях это ожидаемое значение наблюдения в состоянии $\hat{\boldsymbol{\mu}}(t)$

### 2.3 Интерфейс фильтра Калмана<sup>[toc](#toc)</sup>

Интерфейс фильтра Калмана, представленный выше в виде функции `kalman_process_observation`, вполне рассчитан на такой случай. Более того, если при реализации внутри этой функции использована функция `get_ekf_expected_observation`, то вам даже менять ничего не потребуется. Ниже представлены 4 функции, которые нужно будет обновить для нового состояния.

In [ ]:
def get_ekf_transition_matrix(
        ekf_state: EkfState,
        time_step: float) -> np.ndarray:
    # TODO(Implement)
    raise NotImplementedError("Please implement this function")


def get_ekf_transition_noise_cov_matrix(ekf_state: EkfState, time_step: float):
    # TODO(Implement)
    # Так как в задаче переменный шаг, то для получения хорошего результата матрицу шума скорее всего
    # потребуется сделать линейно зависимой от шага. Т.е. получается так, что дисперсия перехода
    # в новое состояние как бы нарастает с некоторой скоростью: чем больше не видим систему,
    # тем больше становится дисперсия её состояния
    raise NotImplementedError("Please implement this function")


def get_ekf_expected_observation(ekf_state: EkfState) -> np.ndarray:
    # TODO(Implement)
    raise NotImplementedError("Please implement this function")


def get_ekf_observation_matrix(ekf_state: EkfState) -> np.ndarray:
    # TODO(Implement)
    raise NotImplementedError("Please implement this function")


def get_ekf_observation_noise_cov_matrix(ekf_state: EkfState) -> np.ndarray:
    # TODO(Implement)
    raise NotImplementedError("Please implement this function")

#### Запускаем моделирование<sup>[toc](#toc)<sup>

In [ ]:
def get_initial_ekf_state(initial_time: T.Optional[float] = None) -> EkfState:
    # Функциональный генератор начального состояния EKF-фильтра
    # Если нужно, поменайте значения начального среднего и ковариации на свое усмотрение
    if initial_time is None:
        initial_time = 0.0
    assert isinstance(initial_time, numbers.Number)
    return EkfState(
        time=float(initial_time),
        size=3,
        mean=np.ones(3),
        cov=np.diag([5., 4., 3.]))


initial_ekf_state = get_initial_ekf_state(initial_time=observations_times[0])
print('Initial EKF state:')
print(f'\ttime: {initial_ekf_state.time}')
print(f'\tsize: {initial_ekf_state.size}')
print(f'\tmean: {initial_ekf_state.mean}')
print(f'\tcov: {initial_ekf_state.cov[np.arange(3), np.arange(3)]}')

In [ ]:
ekf_state = copy.deepcopy(initial_ekf_state)
ekf_states = []

assert len(observations_times) == len(observations)
for observation_time, observation in zip(observations_times, observations):
    assert ekf_state.time <= observation_time
    # Step 1: Transition/prediction
    # moving state to current observation time that is finding
    # prior state distribution for time "observation_time"
    ekf_state = kalman_transit_state(ekf_state, observation_time)
    assert ekf_state.time == observation_time
    
    # Step 2. State correction/observation processing
    # find posterior state distribution for time "observation time"
    observation = np.array([observation], dtype=np.float64)
    ekf_state = kalman_process_observation(ekf_state, observation)

    ekf_states.append(ekf_state)

# Phase, amplitude, frequency (PAF) states
paf_ekf_states = copy.deepcopy(ekf_states)

### Визуализация тректорий $\varphi(t)$, $A(t)$ и $\omega(t)$<sup>[toc](#toc)</sup>

> В случае правильной реализации значения $A(t)$ и $\omega(t)$ будут колебаться около своих истинных значений. Разве что возможно ситуация сдвига фазы на значения кратные $\pi$. Учтите, что при этом знак амплитуды может измениться на противоположный

In [ ]:
phases = [x.mean[0] for x in paf_ekf_states]
amplitudes = [x.mean[1] for x in paf_ekf_states]
angular_velocities = [x.mean[2] for x in paf_ekf_states]

fig, axarr = plt.subplots(nrows=3, ncols=1, sharex=True, figsize=(8, 16))

ax = axarr[0]
ax.plot(observations_times, gt_angular_frequency * observations_times + gt_initial_phase, color='g', label='gt')
ax.plot(observations_times, phases, color='b', label='ekf')
ax.set_title('phase(t)')
ax.legend()

ax = axarr[1]
ax.hlines(gt_amplitude, xmin=observations_times[0], xmax=observations_times[-1], color='g', label='gt')
ax.plot(observations_times, amplitudes, color='b', label='ekf')
ax.set_title('amplitude(t)')
ax.legend()

ax = axarr[2]
ax.hlines(gt_angular_frequency, xmin=observations_times[0], xmax=observations_times[-1], color='g', label='gt')
ax.plot(observations_times, angular_velocities, color='b', label='ekf')
ax.set_title('omega(t)')
ax.legend();

### Визуализация тректорий $x(t)$, $\upsilon(t)$ и $a(t)$<sup>[toc](#toc)</sup>
> В данном случае у вас должно получиться очень хорошее совпадение все трех параметров с GT

In [ ]:
def convert_paf_to_xva_ekf_state(ekf_state) -> EkfState:
    phase, amplitude, angular_frequency = ekf_state.mean
    position = amplitude * np.cos(phase)
    velocity = -angular_frequency * amplitude * np.sin(phase)
    acceleration = -angular_frequency**2 * amplitude * np.cos(phase)
    return EkfState(
        time=ekf_state.time,
        size=3,
        cov=None,
        mean=np.array([position, velocity, acceleration]))

xva_ekf_states = list(map(convert_paf_to_xva_ekf_state, paf_ekf_states))

In [ ]:
# Отрисуем последние 100 значений
num_observations_to_draw = 100

fig, axarr = plt.subplots(nrows=3, ncols=1, sharex=True, figsize=(8, 15))

# Drawing x(t)
gt_xs = gt_observations
obs_xs = observations
ekf_xs = [ekf_state.mean[0] for ekf_state in xva_ekf_states]

ax = axarr[0]
ax.plot(
    observations_times[-num_observations_to_draw:],
    gt_xs[-num_observations_to_draw:], color='g', label='gt')
ax.plot(
    observations_times[-num_observations_to_draw:],
    obs_xs[-num_observations_to_draw:], color='r', label='obs')
ax.plot(
    observations_times[-num_observations_to_draw:],
    ekf_xs[-num_observations_to_draw:], color='b', label='ekf')
ax.set_title('x(t)')
ax.set_ylabel('m')
ax.legend()

# Drawing v(t)
ax = axarr[1]
gt_velocities = -gt_angular_frequency * gt_amplitude * np.sin(
    gt_angular_frequency * observations_times + gt_initial_phase)
ekf_velocities = [ekf_state.mean[1] for ekf_state in xva_ekf_states]

ax.plot(
    observations_times[-num_observations_to_draw:],
    gt_velocities[-num_observations_to_draw:], color='g', label='gt')
ax.plot(
    observations_times[-num_observations_to_draw:],
    ekf_velocities[-num_observations_to_draw:], color='b', label='ekf')

ax.set_title('v(t)')
ax.set_ylabel('m/s')
ax.legend()

# Drawing a(t)
ax = axarr[2]
gt_accelerations = -gt_angular_frequency**2 * gt_amplitude * np.cos(
    gt_angular_frequency * observations_times + gt_initial_phase)
ekf_accelerations = [ekf_state.mean[2] for ekf_state in xva_ekf_states]

ax.plot(
    observations_times[-num_observations_to_draw:],
    gt_accelerations[-num_observations_to_draw:], color='g', label='gt')
ax.plot(
    observations_times[-num_observations_to_draw:],
    ekf_accelerations[-num_observations_to_draw:], color='b', label='ekf')

ax.set_title('a(t)')
ax.set_ylabel('m/s^2')
ax.legend();

### Подсчет MSE<sup>[toc](#toc)</sup>
> MSE должен оказаться сильно меньше, чем в случае первого EKF-фильтра

In [ ]:
np.mean((gt_xs - ekf_xs)**2)